# Simple Tool Approval Policy Engine v1 Showcase

This notebook showcases ORBIT's first implemented policy-group runtime behavior before the MCP phase.

It demonstrates:
- `permission_authority` pressure escalation
- structured runtime-specific reauthorization
- `system_environment` metadata-driven environment decisions
- canonical long-session validation scenarios `PG-PA-01` and `PG-SE-01`


In [ ]:
import sys
from pathlib import Path

ROOT = Path('/Volumes/2TB/MAS/openclaw-core/ORBIT')
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from orbit.runtime.core.session_manager import SessionManager
from orbit.runtime.execution.contracts.plans import ExecutionPlan, ToolRequest
from orbit.runtime.testing.long_session_runner import LongSessionScenario, LongSessionScenarioRunner
from orbit.runtime.governance.tool_approval_policy import PolicyEvaluationInput, evaluate_tool_approval_policy
from orbit.store.sqlite_store import SQLiteStore


## 1. Policy-decision examples


In [ ]:
cases = [
    PolicyEvaluationInput(policy_group='permission_authority', approval_required=True, appearance_count_after_rejection=0, has_structured_reauthorization=False, environment_status='ok', tool_name='write_file'),
    PolicyEvaluationInput(policy_group='permission_authority', approval_required=True, appearance_count_after_rejection=1, has_structured_reauthorization=False, environment_status='ok', tool_name='write_file'),
    PolicyEvaluationInput(policy_group='permission_authority', approval_required=True, appearance_count_after_rejection=2, has_structured_reauthorization=False, environment_status='ok', tool_name='write_file'),
    PolicyEvaluationInput(policy_group='system_environment', approval_required=False, appearance_count_after_rejection=0, has_structured_reauthorization=False, environment_status='denied', tool_name='read_file'),
    PolicyEvaluationInput(policy_group='system_environment', approval_required=False, appearance_count_after_rejection=0, has_structured_reauthorization=False, environment_status='unknown', tool_name='read_file'),
]
[(c.policy_group, c.tool_name, evaluate_tool_approval_policy(c).outcome, evaluate_tool_approval_policy(c).reason) for c in cases]

## 2. Shared runtime helpers


In [ ]:
class StubCoordinator:
    def __init__(self, session_manager):
        self.session_manager = session_manager
    def create_session(self, backend_name, model):
        return self.session_manager.create_session(backend_name=backend_name, model=model)
    def run_session_turn(self, session_id, user_input):
        return self.session_manager.run_session_turn(session_id=session_id, user_input=user_input)
    def list_open_session_approvals(self):
        return self.session_manager.list_open_session_approvals()
    def resolve_session_approval(self, session_id, approval_request_id, decision, note=None):
        return self.session_manager.resolve_session_approval(session_id=session_id, approval_request_id=approval_request_id, decision=decision, note=note)
    def inspect_session(self, session_id):
        session = self.session_manager.get_session(session_id)
        messages = self.session_manager.list_messages(session_id)
        return {'session': session, 'messages': messages}

def fresh_store(name: str):
    path = Path(f'/tmp/{name}.db')
    path.unlink(missing_ok=True)
    return SQLiteStore(db_path=path)


## 3. `permission_authority` runtime path


In [ ]:
class PermissionAuthorityBackend:
    def __init__(self):
        self.calls = 0
    def plan_from_messages(self, messages, session=None):
        self.calls += 1
        return ExecutionPlan(
            source_backend='stub',
            plan_label=f'pa-showcase-{self.calls}',
            tool_request=ToolRequest(
                tool_name='write_file',
                input_payload={'path': 'tmp_policy_showcase.txt', 'content': f'call-{self.calls}'},
                side_effect_class='write',
                requires_approval=True,
            ),
            should_finish_after_tool=False,
        )

store = fresh_store('orbit_pa_showcase')
backend = PermissionAuthorityBackend()
manager = SessionManager(store=store, backend=backend, workspace_root=str(ROOT))
session = manager.create_session(backend_name='stub', model='stub-model')

plan1 = manager.run_session_turn(session_id=session.session_id, user_input='please write the file')
approvals = manager.list_open_session_approvals()
plan2 = manager.resolve_session_approval(session_id=session.session_id, approval_request_id=approvals[0]['approval_request_id'], decision='reject', note='no')
plan3 = manager.run_session_turn(session_id=session.session_id, user_input='do it anyway')
[(m.turn_index, str(m.role), m.metadata.get('message_kind'), m.content) for m in manager.list_messages(session.session_id)]

## 4. Structured runtime-specific reauthorization


In [ ]:
store = fresh_store('orbit_reauth_showcase')
backend = PermissionAuthorityBackend()
manager = SessionManager(store=store, backend=backend, workspace_root=str(ROOT))
session = manager.create_session(backend_name='stub', model='stub-model')

plan1 = manager.run_session_turn(session_id=session.session_id, user_input='please write the file')
approvals = manager.list_open_session_approvals()
_ = manager.resolve_session_approval(session_id=session.session_id, approval_request_id=approvals[0]['approval_request_id'], decision='reject', note='no')
reauth = ToolGovernanceService(manager).reauthorize_tool_path(session_id=session.session_id, tool_name='write_file', note='showcase reauth', source='runtime_cli')
plan2 = manager.run_session_turn(session_id=session.session_id, user_input='try again after reauth')
{'reauth': reauth, 'plan2_label': plan2.plan_label, 'open_approvals': manager.list_open_session_approvals()}

## 5. `system_environment` runtime path


In [ ]:
existing = ROOT / 'tmp_policy_existing.txt'
existing.write_text('hello system environment')

class EnvironmentBackend:
    def __init__(self, request):
        self.request = request
    def plan_from_messages(self, messages, session=None):
        return ExecutionPlan(
            source_backend='stub',
            plan_label='se-showcase',
            tool_request=self.request,
            should_finish_after_tool=False,
        )

def run_env_case(name, request):
    store = fresh_store(f'orbit_env_showcase_{name}')
    manager = SessionManager(store=store, backend=EnvironmentBackend(request), workspace_root=str(ROOT))
    session = manager.create_session(backend_name='stub', model='stub-model')
    plan = manager.run_session_turn(session_id=session.session_id, user_input=name)
    messages = manager.list_messages(session.session_id)
    return plan.plan_label, [(m.turn_index, str(m.role), m.metadata.get('message_kind'), m.content) for m in messages]

existing_case = run_env_case('existing', ToolRequest(tool_name='read_file', input_payload={'path': 'tmp_policy_existing.txt'}, side_effect_class='safe', requires_approval=False))
missing_case = run_env_case('missing', ToolRequest(tool_name='read_file', input_payload={'path': 'tmp_policy_missing.txt'}, side_effect_class='safe', requires_approval=False))
unknown_case = run_env_case('unknown', ToolRequest(tool_name='read_file', input_payload={}, side_effect_class='safe', requires_approval=False))
{'existing': existing_case, 'missing': missing_case, 'unknown': unknown_case}

## 6. Canonical long-session validation scenarios


In [ ]:
coordinator = StubCoordinator(SessionManager(store=fresh_store('orbit_pg_pa_runner'), backend=PermissionAuthorityBackend(), workspace_root=str(ROOT)))
runner = LongSessionScenarioRunner(coordinator)
pa_scenario = LongSessionScenario(
    name='PG-PA-01 permission_authority pressure escalation',
    backend_name='stub',
    model='stub-model',
    turns=['please write the file', 'do it anyway'],
    workspace_root=ROOT,
    approval_decisions={1: 'reject'},
    expected_last_message_kind_by_turn={1: 'policy_caution', 2: 'policy_termination'},
    expect_session_terminated=True,
)
pa_result = runner.run(pa_scenario)
pa_errors = runner.validate_result(pa_scenario, pa_result)
{'scenario': pa_result.scenario_name, 'terminated': pa_result.session_terminated, 'termination_reason': pa_result.termination_reason, 'errors': pa_errors}

In [ ]:
existing = ROOT / 'tmp_pg_se_nb_existing.txt'
existing.write_text('hello se canonical notebook')

def run_pg_se_case(label, request, expected_kind):
    coordinator = StubCoordinator(SessionManager(store=fresh_store(f'orbit_pg_se_runner_{label}'), backend=EnvironmentBackend(request), workspace_root=str(ROOT)))
    runner = LongSessionScenarioRunner(coordinator)
    scenario = LongSessionScenario(
        name=f'PG-SE-01 {label}',
        backend_name='stub',
        model='stub-model',
        turns=[label],
        workspace_root=ROOT,
        expected_last_message_kind_by_turn={1: expected_kind},
        expect_session_terminated=False,
    )
    result = runner.run(scenario)
    return {
        'scenario': result.scenario_name,
        'plan_label': result.turns[0].plan['plan_label'],
        'last_kind': result.turns[0].last_message.get('metadata', {}).get('message_kind') if result.turns[0].last_message else None,
        'errors': runner.validate_result(scenario, result),
    }

pg_se_existing = run_pg_se_case('existing', ToolRequest(tool_name='read_file', input_payload={'path': 'tmp_pg_se_nb_existing.txt'}, side_effect_class='safe', requires_approval=False), 'tool_result')
pg_se_missing = run_pg_se_case('missing', ToolRequest(tool_name='read_file', input_payload={'path': 'tmp_pg_se_nb_missing.txt'}, side_effect_class='safe', requires_approval=False), 'policy_decision')
pg_se_unknown = run_pg_se_case('unknown', ToolRequest(tool_name='read_file', input_payload={}, side_effect_class='safe', requires_approval=False), 'policy_decision')
{'existing': pg_se_existing, 'missing': pg_se_missing, 'unknown': pg_se_unknown}

## 7. Takeaways before MCP

This showcase demonstrates that ORBIT now has:
- first-pass runtime truth for `permission_authority`
- first-pass runtime truth for `system_environment`
- metadata-driven tool governance semantics that can carry forward into MCP
- canonical long-session validation scenarios for both first-pass Policy Groups

That makes the governance-policy boundary substantially clearer before re-entering the MCP development phase.
